# Capstone (No Key) - Small Business Finance Assistant (Solution)
**Day 3 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/capstone_nokey/solution/capstone_nokey.ipynb)

**Goal:** Build a retrieval-augmented assistant that answers small-business
finance questions over a mock QuickBooks and TurboTax help knowledge base,
grounded in the retrieved help articles. Then extend it with an agentic tool
loop, an LLM-as-a-judge evaluation, a retrieval comparison, and a LoRA-trained
ticket router.

> **How this runs without an API key.** This is the no-key version of the capstone. Retrieval and fine-tuning already run offline. For generation and the judge it swaps the hosted LLM for a small local model, `google/flan-t5-base`, that downloads once (about 1 GB) and runs on CPU. The agentic stretch replaces the LLM planner with a scripted planner so the tool loop stays visible and deterministic. Everything else is identical to the API capstone; only the call sites change (`generate(...)` and a scripted planner instead of `get_completion(...)` and a `LiteLLMModel` agent).

All data in `capstone/data/` is synthetic mock data. This solution has the full
working code. No API key needed. Runs on CPU; no GPU needed.

In [ ]:
# Scoped install for this capstone (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers faiss-cpu transformers torch peft accelerate

## Setup: data path and a local model
This finds the `capstone/data/` folder locally, or clones the companion repo
when running on Colab, then loads a small local instruction model for
generation. No provider key and no shared client. You do not need to change
this cell.

In [ ]:
import os
import subprocess
import sys

import torch  # noqa: F401  (import torch before faiss to avoid an OpenMP clash on macOS)


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "capstone", "data")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for the data files.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
DATA_DIR = os.path.join(root, "capstone", "data")

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/flan-t5-base"  # use "google/flan-t5-small" for a smaller download
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
llm.eval()


def generate(prompt, max_new_tokens=96):
    """Greedy text-in, text-out call to the local model (deterministic)."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = llm.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print("Loaded", MODEL_NAME, "and data at", DATA_DIR)

## 1. Load the knowledge base
Read every Markdown help article from `capstone/data/`. Each file is one
source document the assistant is allowed to use.

In [ ]:
import glob

documents = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.md"))):
    with open(path, encoding="utf-8") as f:
        documents.append({"source": os.path.basename(path), "text": f.read()})
print(f"Loaded {len(documents)} help articles:",
      ", ".join(d["source"] for d in documents))

## 2. Chunk, embed, and index
Split each article into overlapping chunks, embed them with a small local
model, and index the vectors with FAISS. Normalized vectors make inner-product
search equal to cosine similarity.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


def chunk_text(text, chunk_size=500, overlap=60):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


def build_index(docs, model, chunk_size=500, overlap=60):
    chunks = []
    for doc in docs:
        for piece in chunk_text(doc["text"], chunk_size, overlap):
            chunks.append({"source": doc["source"], "text": piece})
    vecs = np.array(model.encode([c["text"] for c in chunks],
                                 normalize_embeddings=True), dtype="float32")
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return chunks, idx


embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunks, index = build_index(documents, embedder)
print(f"Indexed {index.ntotal} chunks.")

## 3. Retrieve and answer, grounded
Retrieve the top chunks for a question, gate on whether they actually answer it,
then answer using only that context and cite the source article. If the context
does not contain the answer, the assistant declines.

In [ ]:
INSTRUCTIONS = (
    "Answer the question using only the context from the QuickBooks and TurboTax "
    "help articles. Keep the answer short."
)


def retrieve(query, model, idx, chunk_list, k=3):
    q = np.array(model.encode([query], normalize_embeddings=True), dtype="float32")
    scores, ids = idx.search(q, k)
    hits = []
    for score, i in zip(scores[0], ids[0]):
        hit = dict(chunk_list[i])
        hit["score"] = float(score)
        hits.append(hit)
    return hits


def grounded(query, context):
    """Grounding gate: ask the local model whether the context answers the question.

    A hosted model can be told to decline in the answer prompt. A small local model
    answers a plain yes/no question more reliably, so we gate on that first. This
    keeps the assistant from making up an answer when the fact is not in the KB.
    """
    verdict = generate(
        f"Context: {context}\n\nCan the question be answered using only the context "
        f"above? Question: {query}\nAnswer yes or no."
    )
    return verdict.strip().lower().startswith("y")


def rag_answer(query, k=3):
    hits = retrieve(query, embedder, index, chunks, k)
    context = "\n\n".join(h["text"] for h in hits)
    if not grounded(query, context):
        return "I do not have that information in the knowledge base.", hits
    answer = generate(f"{INSTRUCTIONS}\nContext: {context}\nQuestion: {query}").strip()
    return f"{answer} [{hits[0]['source']}]", hits


DEMO_QUESTIONS = [
    "What is the default payment term on a QuickBooks invoice?",
    "How much does QuickBooks Payroll Core cost?",
    "Can I deduct a home office, and how much?",
    "Does QuickBooks integrate with Salesforce?",  # not in the KB
]
for q in DEMO_QUESTIONS:
    ans, _ = rag_answer(q)
    print(f"\nQ: {q}\nA: {ans}")

## Stretch 1: agentic RAG with a computation tool
RAG retrieves facts but does not compute. We give an agent two tools, one to
search the knowledge base and one to compute an invoice total with sales tax,
and run the tool loop by hand with a scripted planner (the no-key stand-in for
the model that would otherwise choose each step).

In [ ]:
TAX_RATES = {"CA": 7.25, "NY": 4.0, "TX": 6.25, "WA": 6.5, "FL": 6.0}


def search_help(query: str) -> str:
    """Tool: search the QuickBooks and TurboTax help knowledge base."""
    hits = retrieve(query, embedder, index, chunks, k=2)
    return "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)


def invoice_total(subtotal: float, state_code: str) -> float:
    """Tool: compute an invoice total including state sales tax."""
    rate = TAX_RATES.get(state_code.upper(), 0.0)
    return round(subtotal * (1 + rate / 100.0), 2)


TOOLS = {"search_help": search_help, "invoice_total": invoice_total}


def plan_next_step(question, observations):
    """Scripted stand-in for the LLM planner (deterministic).

    A real agent lets the model choose the next tool from the question and the
    observations so far. Here we script it so the tool loop is fully visible.
    """
    if len(observations) == 0:
        return {"type": "action",
                "thought": "Search the KB for the California sales tax rate.",
                "tool": "search_help",
                "args": {"query": "California sales tax rate"}}
    if len(observations) == 1:
        return {"type": "action",
                "thought": "Now compute the 1200 dollar invoice total with CA tax.",
                "tool": "invoice_total",
                "args": {"subtotal": 1200.0, "state_code": "CA"}}
    return {"type": "final",
            "answer": f"The invoice total including California sales tax is ${observations[1]:.2f}."}


def run_agent(question, max_steps=5):
    print(f"Goal: {question}\n")
    observations = []
    for step in range(1, max_steps + 1):
        decision = plan_next_step(question, observations)
        if decision["type"] == "final":
            return decision["answer"]
        print(f"Step {step}")
        print(f"  Thought    : {decision['thought']}")
        print(f"  Action     : {decision['tool']}({decision['args']})")
        observation = TOOLS[decision["tool"]](**decision["args"])
        preview = str(observation)
        print(f"  Observation: {preview[:80]}{'...' if len(preview) > 80 else ''}")
        observations.append(observation)
    return "Stopped: hit the step limit."


print("\nFinal answer:", run_agent(
    "A customer in California owes 1200 dollars before tax. What is the invoice "
    "total including sales tax?"
))

## Stretch 2: evaluate with an LLM-as-a-judge
We score the assistant on the golden question and answer pairs in
`golden_qa.json`. The same local model acts as the judge: for each pair it
decides yes or no whether the answer agrees with the expected answer.

In [ ]:
import json

with open(os.path.join(DATA_DIR, "golden_qa.json"), encoding="utf-8") as f:
    golden = json.load(f)


def judge(question, answer, expected):
    """Local LLM-as-a-judge: ask the model yes/no if the answer matches expected."""
    verdict = generate(
        f"Question: {question}\nExpected answer: {expected}\nModel answer: {answer}\n"
        f"Does the model answer agree with the expected answer? Answer yes or no."
    )
    return verdict.strip().lower().startswith("y")


passed = 0
for case in golden:
    ans, _ = rag_answer(case["question"])
    ok = judge(case["question"], ans, case["expected"])
    passed += int(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {case['question']}\n      -> {ans}")

print(f"\nScore: {passed}/{len(golden)} judged correct.")

## Stretch 3: compare chunking strategies
Retrieval quality depends on how you chunk. We rebuild the index with smaller
chunks and compare which source article each question retrieves for the same
questions. This stretch is fully offline.

In [ ]:
small_chunks, small_index = build_index(documents, embedder, chunk_size=200, overlap=30)
print(f"Default chunks: {len(chunks)}   small chunks: {len(small_chunks)}\n")

for q in [c["question"] for c in golden[:3]]:
    default_src = retrieve(q, embedder, index, chunks, k=1)[0]["source"]
    small_src = retrieve(q, embedder, small_index, small_chunks, k=1)[0]["source"]
    match = "same" if default_src == small_src else "different"
    print(f"[{match}] {q}\n   default -> {default_src}   small -> {small_src}")

## Stretch 4: LoRA-train a support-ticket router
We adapt the LoRA idea from Lab 3.2 to a classifier: a DistilBERT encoder with
a LoRA adapter learns to route tickets into billing, payroll, tax_filing, or
integrations. Only the adapter and the head train. This stretch is fully
offline.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)
from torch.utils.data import Dataset

with open(os.path.join(DATA_DIR, "support_tickets.json"), encoding="utf-8") as f:
    tickets = json.load(f)

LABELS = ["billing", "payroll", "tax_filing", "integrations"]
label2id = {name: i for i, name in enumerate(LABELS)}
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")


class TicketDataset(Dataset):
    def __init__(self, rows):
        self.enc = tok([r["text"] for r in rows], padding=True,
                       truncation=True, max_length=48)
        self.labels = [label2id[r["label"]] for r in rows]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


clf = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(LABELS),
    id2label={i: n for n, i in label2id.items()}, label2id=label2id)
clf = get_peft_model(clf, LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_lin", "v_lin"],
    task_type="SEQ_CLS"))
clf.print_trainable_parameters()

Train the router for a handful of epochs on the 16 example tickets, then
route the held-out test tickets.

In [ ]:
trainer = Trainer(
    model=clf,
    args=TrainingArguments(output_dir="ticket-router", num_train_epochs=30,
                           per_device_train_batch_size=8, learning_rate=5e-4,
                           logging_steps=20, report_to="none", use_cpu=True,
                           seed=42),
    train_dataset=TicketDataset(tickets["train"]),
)
trainer.train()

clf.eval()
correct = 0
for row in tickets["test"]:
    enc = tok(row["text"], return_tensors="pt", truncation=True, max_length=48)
    with torch.no_grad():
        pred = clf(**enc).logits.argmax(-1).item()
    routed = clf.config.id2label[pred]
    correct += int(routed == row["label"])
    print(f"[{routed:>12} | true {row['label']:>12}]  {row['text']}")
print(f"\nRouting accuracy: {correct}/{len(tickets['test'])}")

## Expected output
- The base assistant answers the finance questions with a source citation and
  declines the Salesforce question because the fact is not in the KB.
- The agentic stretch searches the KB and computes a 1200 dollar California
  invoice total of 1287.00.
- The judge marks most of the golden Q&A set correct (about 4 or 5 of 5). A
  small local model is stricter and terser than a hosted one, so an occasional
  miss on a terse answer is expected.
- The chunking comparison shows which questions retrieve the same source under
  both chunk sizes.
- The LoRA ticket router trains only a small adapter and routes most or all of
  the four held-out tickets correctly.